<div class="alert alert-block alert-success">

<b>CODING BACKPROPAGATION FROM SCRATCH: ON A SINGLE NEURON</b>
</div>

In [1]:
import numpy as np 

# initialize parameters 
inputs =  np.array([1.0, -2.0, 3.0])
weights = np.array([-3.0, -1.0, 2.0])
bias = 1.0
target_output = 0
learning_rate = 0.001

def relu(x):
    return np.maximum(x, 0)

def relu_derivative(x):
    return np.where(x>0, 1.0, 0)

for i in range(20):
    # Forward pass 
    sum = np.dot(inputs, weights) + bias
    output = relu(sum)
    loss = (output - target_output)**2

    # Backward pass
    dloss_doutput = 2 * (output - target_output)
    doutput_drelu = relu_derivative(sum)
    dmul_dwi = inputs
    dmul_dbias = 1.0

    dloss_dwi = dloss_doutput * doutput_drelu * dmul_dwi
    dloss_dbi = dloss_doutput * doutput_drelu * dmul_dbias

    # Update the weights and bias
    weights = weights - learning_rate * dloss_dwi
    bias = bias - learning_rate * dloss_dbi

    # Print the loss for this iteration
    print(f"Iteration {i+ 1}, Loss: {loss}")

print(f"Final Loss {loss}")
print(f"Final Bias {bias}")


Iteration 1, Loss: 36.0
Iteration 2, Loss: 33.872399999999985
Iteration 3, Loss: 31.870541159999995
Iteration 4, Loss: 29.98699217744401
Iteration 5, Loss: 28.21476093975706
Iteration 6, Loss: 26.54726856821742
Iteration 7, Loss: 24.978324995835766
Iteration 8, Loss: 23.502105988581878
Iteration 9, Loss: 22.113131524656684
Iteration 10, Loss: 20.80624545154949
Iteration 11, Loss: 19.576596345362915
Iteration 12, Loss: 18.419619501351963
Iteration 13, Loss: 17.331019988822064
Iteration 14, Loss: 16.306756707482677
Iteration 15, Loss: 15.343027386070442
Iteration 16, Loss: 14.43625446755368
Iteration 17, Loss: 13.583071828521266
Iteration 18, Loss: 12.780312283455652
Iteration 19, Loss: 12.024995827503426
Iteration 20, Loss: 11.314318574097976
Final Loss 11.314318574097976
Final Bias 0.8175177371706989


<div class="alert alert-block alert-success">
<b>BACKPROPAGATION </b>
</div>

<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO WEIGHTS
</div>

In [2]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# We have 3 sets of inputs - samples
inputs = np.array([[1, 2, 3, 2.5],
 [2., 5., -1., 2],
 [-1.5, 2.7, 3.3, -0.8]])
# sum weights of given input
# and multiply by the passed-in gradient for this neuron

dweights = np.dot(inputs.T, dvalues)
print(dweights)

[[ 0.5  0.5  0.5]
 [20.1 20.1 20.1]
 [10.9 10.9 10.9]
 [ 4.1  4.1  4.1]]


<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO BIASES
</div>

In [3]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# One bias for each neuron
# biases are the row vector with a shape (1, neurons)
biases = np.array([[2, 3, 0.5]])
# dbiases - sum values, do this over samples (first axis), keepdims
# since this by default will produce a plain list -

dbiases = np.sum(dvalues, axis= 0, keepdims= True)
print(dbiases)


[[6. 6. 6.]]


<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO INPUTS
</div>

In [4]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# We have 3 sets of weights - one set for each neuron
# we have 4 inputs, thus 4 weights
# recall that we keep weights transposed
weights = np.array([[0.2, 0.8, -0.5, 1],
 [0.5, -0.91, 0.26, -0.5],
 [-0.26, -0.27, 0.17, 0.87]]).T
# sum weights of given input
# and multiply by the passed-in gradient for this neuron

dinputs = np.dot(dvalues, weights.T)
print(dinputs)

[[ 0.44 -0.38 -0.07  1.37]
 [ 0.88 -0.76 -0.14  2.74]
 [ 1.32 -1.14 -0.21  4.11]]


## Definition of Layer with backpropagation

In [5]:
class Layer:
    def __init__(self, row, cols):
        self.weights = 0.01 * np.random.randn(row, cols)
        self.bias = np.zeros((1, cols))

    def forward(self, inputs):
        self.x = inputs
        self.output = np.dot(self.x, self.weights) + self.bias
        return self.output

    def backward(self, dvalues):
        self.dL_dz = dvalues
        self.dweights = np.dot(self.x.T, self.dL_dz  )
        self.dbias = np.sum(self.dL_dz, axis = 0, keepdims= True)
        self.dinputs = np.dot(self.dL_dz, self.weights.T)

## Definition of ReLU activation function with backpropagation


In [6]:
class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

## Definition of Categorical Cross Entropy Loss

In [7]:
class CrossEntropyLoss:
    def forward(self, y_pred, y_true):
        y_pred_cliped = np.clip(y_pred, 1e-5, 1-1e-5 )
        # check for one hot encoding 
        if (len(y_true.shape)== 1):
            # Not One hot encoded 
            y_true = np.eye(len(y_pred[0]))[y_true]

        
        correct_confidences = np.sum(y_true * np.log(y_pred_cliped), axis=1, keepdims= True)

        neg_loss = - correct_confidences
        avg_loss = np.mean(neg_loss)
        return avg_loss

    def backward(self, dvalues , y_true):
        if (len(y_true.shape)== 1):
            y_true = np.eye(len(dvalues[0]))[y_true]

        dvalues = np.clip(dvalues, 1e-5, 1-1e-5)
        
        self.dinputs = - y_true/dvalues
        self.dinputs = self.dinputs/(len(dvalues))

        
        

In [8]:
# Softmax activation
class Activation_Softmax:
 # Forward pass
 def forward(self, inputs):
 # Get unnormalized probabilities
  exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
 # Normalize them for each sample
  probabilities = exp_values / np.sum(exp_values, axis=1,keepdims=True)
  self.output = probabilities

## Definition of Softmax loss backpropagation

In [9]:
class Activation_Softmax_crossEntropy_loss:
    def __init__(self):
        self.softmax = Activation_Softmax()
        self.cross = CrossEntropyLoss()

    def forward(self, inputs, y_true):
        self.softmax.forward(inputs)
        self.outputs = self.cross.forward(self.softmax.output, y_true)
        return self.outputs

    def backward(self, predicted , y_true):
        samples = len(predicted)
        # if the y_true is one hot encoded
        if(len(y_true.shape) == 2):
            y_true = np.argmax(y_true, axis = 1)
        
        self.dinputs = predicted.copy()
        self.dinputs[range(len(predicted)), y_true] -= 1
        self.dinputs = self.dinputs/samples

In [10]:
softmax_outputs = np.array([
    [0.7, 0.1, 0.2],
    [0.1, 0.5, 0.4],
    [0.02, 0.9, 0.08]
])

y_true = np.array([0, 1, 1])

softmax = Activation_Softmax_crossEntropy_loss()
softmax.backward(softmax_outputs, y_true)
dvalues = softmax.dinputs
print("Gradient: combined loss and activation func:", dinputs)


Gradient: combined loss and activation func: [[ 0.44 -0.38 -0.07  1.37]
 [ 0.88 -0.76 -0.14  2.74]
 [ 1.32 -1.14 -0.21  4.11]]


# ASSAMBLE EVERYTHING TO MAKE A NEURAL NETWORK

In [11]:
from nnfs.datasets import spiral_data


<img src = 'attachments/full_nn.jpeg' alt = 'dense layer example' width = 'auto' height = 'auto'>

In [12]:

X, y = spiral_data(samples=100, classes=3)

# first Layer 2 inputs -> 3 neuron
dense1 = Layer(2, 3)

# ReLU layer
activation1 = relu()

# secend dense layer 3 inputs -> 3 neuron
dense2 = Layer(3, 3)


# softmax and categorical cross entropy loss
loss = Activation_Softmax_crossEntropy_loss()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss.forward(dense2.output, y)

print(f'loss = {loss.outputs}')

predictions = np.argmax(loss.softmax.output, axis = 1)
if len(y.shape) ==2:
    y = np.argmax(y, axis = 1)

acc = np.mean(predictions == y)
print(f"Accuracy= {acc}")

# Backward pass 
loss.backward(loss.softmax.output, y)
dense2.backward(loss.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

# Print gradients
print(dense1.dweights)
print(dense1.dbias)
print(dense2.dweights)
print(dense2.dbias)



loss = 1.0986134675872539
Accuracy= 0.2633333333333333
[[ 1.82082788e-05 -1.39906554e-05  1.97880117e-04]
 [ 1.45420642e-04 -6.29997192e-06 -1.57478850e-04]]
[[-0.00025154  0.0002242   0.00028629]]
[[ 4.77515589e-05  6.06487940e-05 -1.08400353e-04]
 [ 6.45696651e-05 -1.98875614e-04  1.34305949e-04]
 [-9.90477759e-05  1.81042364e-04 -8.19945876e-05]]
[[-1.56807130e-05  5.77317686e-06  9.90753616e-06]]


In [13]:
class Optimization_SGD:
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate

    def update_para(self, layer):
        layer.weights += - layer.dweights * self.learning_rate
        layer.bias += - layer.dbias * self.learning_rate


In [14]:
X, y = spiral_data(samples = 100, classes = 3)

# first Layer 2 inputs -> 64 neuron
dense1 = Layer(2, 128)

# ReLU layer
activation1 = relu()

# secend dense layer 64 inputs -> 3 neuron
dense2 = Layer(128, 3)


# softmax and categorical cross entropy loss
loss = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0)

for epoch in range(10001):
    
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    data_loss = loss.forward(dense2.output, y)


    predictions = np.argmax(loss.softmax.output, axis = 1)
    if len(y.shape) ==2:
        y_true_1d = np.argmax(y, axis = 1)
    else:
        y_true_1d = y

    acc = np.mean(predictions == y_true_1d)
    if (epoch % 100 == 0):
        print(f"iteration = {epoch}, Accuracy= {acc}, loss = {data_loss}")

    # Backward pass 
    loss.backward(loss.softmax.output, y)
    dense2.backward(loss.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # Optimizer: Gradient Descent
    optimizer.update_para(layer= dense1)
    optimizer.update_para(layer = dense2)




iteration = 0, Accuracy= 0.24666666666666667, loss = 1.0986379230940853
iteration = 100, Accuracy= 0.4166666666666667, loss = 1.066273106292916
iteration = 200, Accuracy= 0.4033333333333333, loss = 1.059053764757755
iteration = 300, Accuracy= 0.45, loss = 1.0550573573527586
iteration = 400, Accuracy= 0.44666666666666666, loss = 1.050246467425923
iteration = 500, Accuracy= 0.45666666666666667, loss = 1.0445693537590959
iteration = 600, Accuracy= 0.45666666666666667, loss = 1.0382959241184175
iteration = 700, Accuracy= 0.4766666666666667, loss = 1.0316171839942463
iteration = 800, Accuracy= 0.42333333333333334, loss = 1.028157354499955
iteration = 900, Accuracy= 0.41, loss = 1.021326665270595
iteration = 1000, Accuracy= 0.42, loss = 1.0101015751395725
iteration = 1100, Accuracy= 0.48, loss = 1.0059998845439069
iteration = 1200, Accuracy= 0.47333333333333333, loss = 0.9954494710399728
iteration = 1300, Accuracy= 0.49333333333333335, loss = 0.9738656289233463
iteration = 1400, Accuracy= 0.

In [19]:
class Optimization_SGD:
    def __init__(self, learning_rate = 1., decay = 0.):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = self.learning_rate
        self.iteration = 0

    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate/(1. + self.decay * self.iteration)
    
    def update_para(self, layer):
        layer.weights += - self.current_learning_rate * layer.dweights
        layer.bias += -self.current_learning_rate * layer.dbias

    def post_update(self):
        self.iteration += 1

In [21]:
X , y = spiral_data(samples = 100, classes = 3)

In [26]:


dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_activation = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0 , decay = 0.001)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.softmax.output , axis = 1)
    if (len(y.shape) == 2):
        y_true_1d = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y_true_1d)

    if (epoch%100 == 0):
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")
    
    # calculating gradients 
    loss_activation.backward(loss_activation.softmax.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # optimize the network
    optimizer.pre_update()
    optimizer.update_para(dense1)
    optimizer.update_para(dense2)
    optimizer.post_update()


iteration = 0, accuracy = 0.3433333333333333, loss = 1.0985968622511837, lr = 1.0
iteration = 100, accuracy = 0.43666666666666665, loss = 1.0661235549376078, lr = 0.9099181073703367
iteration = 200, accuracy = 0.4266666666666667, loss = 1.0559514370739422, lr = 0.8340283569641367
iteration = 300, accuracy = 0.43666666666666665, loss = 1.0548587760368255, lr = 0.7698229407236336
iteration = 400, accuracy = 0.45, loss = 1.053883431090809, lr = 0.7147962830593281
iteration = 500, accuracy = 0.4533333333333333, loss = 1.052672572549643, lr = 0.66711140760507
iteration = 600, accuracy = 0.4533333333333333, loss = 1.0513941039339325, lr = 0.6253908692933083
iteration = 700, accuracy = 0.45, loss = 1.049884375615875, lr = 0.5885815185403178
iteration = 800, accuracy = 0.44666666666666666, loss = 1.0481168490787418, lr = 0.5558643690939411
iteration = 900, accuracy = 0.44666666666666666, loss = 1.046042029189306, lr = 0.526592943654555
iteration = 1000, accuracy = 0.45, loss = 1.04354130157888